# Generacion de Modelo Personalizado | StyleGAN2-ADA*

Este cuaderno lo utilizaremos para cargar nuestro modelo entrenado previamente (archivo .pkl) y generar imágenes con diversos parámetros de ajuste. Tambien podremos obtener imágenes en secuencia a modo de interpolación del "espacio latente" para visualizar estos movimientos.

In [1]:
#@title 1. Conectar Colab a  Google Drive
#@markdown Accede a tu Google Drive para cargar tu conjunto de datos, editar las imágenes y guardar los resultados.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#@title 2. Instalación de StyleGAN2-ADA
#@markdown Esta celda prepara StyleGAN2-ADA para la versión actual de Google Colab.
#@markdown
#@markdown El código se instala temporalmente en Colab para obtener mejor rendimiento.
#@markdown Tus datasets, modelos y resultados se guardarán en Google Drive y no se perderán al cerrar la sesión.
#@markdown
#@markdown Ejecuta esta celda una vez al comenzar cada nueva sesión de Colab.

import os
import sys
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

# ------------------------------------------------------------
# 1. Conectar Google Drive
# ------------------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Crear carpetas persistentes en Google Drive
# ------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/colab-sg2-ada-pytorch")

DATASETS_DIR   = DRIVE_ROOT / "datasets"
RESULTS_DIR    = DRIVE_ROOT / "results"
PRETRAINED_DIR = DRIVE_ROOT / "pretrained"

for folder in [DATASETS_DIR, RESULTS_DIR, PRETRAINED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Instalar dependencias necesarias
# ------------------------------------------------------------

print("Instalando dependencias...")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "ninja",
        "imageio-ffmpeg",
        "click",
        "requests",
        "tqdm"
    ],
    check=True
)

# ------------------------------------------------------------
# 4. Descargar StyleGAN2-ADA
# ------------------------------------------------------------

REPO_DIR = Path("/content/stylegan2-ada-pytorch")
REPO_URL = "https://github.com/angelv-salazar/stylegan2-ada-pytorch.git"
BRANCH = "colab-2026"

# El repositorio dentro de /content es temporal.
# Lo descargamos limpio al iniciar cada sesión.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print("Descargando StyleGAN2-ADA...")

subprocess.run(
    [
        "git", "clone",
        "--branch", BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR)
    ],
    check=True
)

os.chdir(REPO_DIR)

# ------------------------------------------------------------
# 5. Verificación
# ------------------------------------------------------------

import torch

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    text=True
).strip()

print("\n----------------------------------------")
print("StyleGAN2-ADA instalado correctamente")
print("----------------------------------------")
print("Repositorio:", REPO_DIR)
print("Rama:", branch)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("\n✅ Todo listo para entrenar.")
else:
    print("\n⚠️ No se detectó una GPU.")
    print("Activa una GPU desde Entorno de ejecución > Cambiar tipo de entorno de ejecución.")

Instalando dependencias...
Descargando StyleGAN2-ADA...

----------------------------------------
StyleGAN2-ADA instalado correctamente
----------------------------------------
Repositorio: /content/stylegan2-ada-pytorch
Rama: colab-2026
PyTorch: 2.11.0+cu130
CUDA: 13.0
GPU: Tesla T4

✅ Todo listo para entrenar.


In [28]:
#@title 5. Generación de Imágenes StyleGAN2

import os
import re
import sys
import warnings
import contextlib

import numpy as np
import torch

from pathlib import Path
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets

import dnnlib
import legacy


# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

warnings.filterwarnings("ignore")


@contextlib.contextmanager
def suppress_output():

    with open(os.devnull, "w") as devnull:

        old_stdout = sys.stdout
        old_stderr = sys.stderr

        sys.stdout = devnull
        sys.stderr = devnull

        try:
            yield

        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


# ============================================================
# INTERFAZ
# ============================================================

project_dir_widget = widgets.Text(
    value="/content/drive/MyDrive/colab-sg2-ada-pytorch/results/amonites",
    description="Proyecto:",
    layout=widgets.Layout(width="95%")
)


network_widget = widgets.Text(
    value="",
    description="Modelo .pkl:",
    layout=widgets.Layout(width="95%")
)


find_latest_button = widgets.Button(
    description="Buscar último snapshot",
    button_style="info"
)


outdir_widget = widgets.Text(
    value="/content/drive/MyDrive/colab-sg2-ada-pytorch/generated",
    description="Salida:",
    layout=widgets.Layout(width="95%")
)


trunc_widget = widgets.FloatSlider(
    value=0.8,
    min=0.1,
    max=1.5,
    step=0.05,
    description="Trunc ψ:",
    continuous_update=False,
    layout=widgets.Layout(width="60%")
)


noise_widget = widgets.Dropdown(
    options=[
        "const",
        "random",
        "none"
    ],
    value="const",
    description="Noise:"
)


seeds_widget = widgets.Text(
    value="1200-1202",
    description="Seeds:",
    layout=widgets.Layout(width="70%")
)


gen_random_widget = widgets.Checkbox(
    value=False,
    description="Generar aleatorias"
)


n_imgs_widget = widgets.IntSlider(
    value=3,
    min=1,
    max=50,
    step=1,
    description="Cantidad:",
    continuous_update=False,
    layout=widgets.Layout(width="60%")
)


save_widget = widgets.Checkbox(
    value=True,
    description="Guardar PNGs"
)


preview_widget = widgets.Checkbox(
    value=True,
    description="Mostrar vista previa"
)


preview_size_widget = widgets.IntSlider(
    value=250,
    min=100,
    max=400,
    step=25,
    description="Preview px:",
    continuous_update=False,
    layout=widgets.Layout(width="60%")
)


# ------------------------------------------------------------
# Formato no cuadrado
# ------------------------------------------------------------

gen_nonsquare_widget = widgets.Checkbox(
    value=False,
    description="Generar formato no cuadrado"
)


width_widget = widgets.IntText(
    value=640,
    description="Ancho:"
)


height_widget = widgets.IntText(
    value=480,
    description="Alto:"
)


generate_button = widgets.Button(
    description="Generar imágenes",
    button_style="success"
)


output_area = widgets.Output()


# ============================================================
# ACTIVAR / DESACTIVAR CONTROLES
# ============================================================

def toggle_n_imgs(change):

    n_imgs_widget.disabled = not change["new"]


gen_random_widget.observe(
    toggle_n_imgs,
    names="value"
)

toggle_n_imgs({
    "new": gen_random_widget.value
})


def toggle_nonsquare(change):

    disabled = not change["new"]

    width_widget.disabled = disabled
    height_widget.disabled = disabled


gen_nonsquare_widget.observe(
    toggle_nonsquare,
    names="value"
)

toggle_nonsquare({
    "new": gen_nonsquare_widget.value
})


# ============================================================
# LEER SEEDS
# ============================================================

def parse_seeds(seeds_str):

    seeds_str = (
        seeds_str
        .strip()
        .replace(" ", "")
    )

    if seeds_str == "":

        raise ValueError(
            "Debes ingresar al menos un seed."
        )


    # --------------------------------------------------------
    # Rango: 100-110
    # --------------------------------------------------------

    if (
        "-" in seeds_str
        and "," not in seeds_str
    ):

        start, end = [
            int(x)
            for x
            in seeds_str.split("-")
        ]

        if end < start:

            raise ValueError(
                "El rango de seeds es inválido."
            )

        return list(
            range(
                start,
                end + 1
            )
        )


    # --------------------------------------------------------
    # Lista: 10,25,40
    # También acepta un solo seed.
    # --------------------------------------------------------

    return [
        int(s)
        for s
        in seeds_str.split(",")
        if s != ""
    ]


# ============================================================
# BUSCAR ÚLTIMO SNAPSHOT
# ============================================================

def find_latest_snapshot(project_dir):

    project_dir = Path(
        project_dir
    )

    if not project_dir.exists():
        return None

    snapshots = list(
        project_dir.glob(
            "**/network-snapshot-*.pkl"
        )
    )

    if not snapshots:
        return None


    def get_rank(path):

        kimg_match = re.search(
            r"network-snapshot-(\d+)\.pkl$",
            path.name
        )

        kimg = (
            int(kimg_match.group(1))
            if kimg_match
            else -1
        )

        run_match = re.match(
            r"^(\d+)-",
            path.parent.name
        )

        run_id = (
            int(run_match.group(1))
            if run_match
            else -1
        )

        return (
            kimg,
            run_id
        )


    return str(
        max(
            snapshots,
            key=get_rank
        )
    )


def find_latest_snapshot_action(b):

    with output_area:

        clear_output(wait=True)

        project_dir = (
            project_dir_widget
            .value
            .strip()
        )

        latest = find_latest_snapshot(
            project_dir
        )

        if latest is None:

            print(
                "No se encontró ningún snapshot "
                "en esa carpeta."
            )

        else:

            network_widget.value = latest

            print(
                "Último snapshot encontrado:"
            )

            print(latest)


find_latest_button.on_click(
    find_latest_snapshot_action
)


# ============================================================
# VISTA PREVIA
# ============================================================

def make_preview_image(
    img,
    preview_size=250
):

    preview = img.copy()

    preview.thumbnail(
        (
            preview_size,
            preview_size
        ),
        Image.LANCZOS
    )

    return preview


# ============================================================
# TENSOR -> PIL
# ============================================================

def tensor_to_pil(
    img_tensor
):

    img = (
        (
            img_tensor.clamp(
                -1,
                1
            )
            + 1
        )
        * 127.5
    )

    img = (
        img
        .permute(
            0,
            2,
            3,
            1
        )
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

    return Image.fromarray(
        img[0],
        "RGB"
    )


# ============================================================
# LABEL
# ============================================================

def make_label(
    G,
    device
):

    if getattr(
        G,
        "c_dim",
        0
    ) > 0:

        return torch.zeros(
            [
                1,
                G.c_dim
            ],
            device=device
        )

    return None


# ============================================================
# CARGAR GENERADOR
# ============================================================

def load_generator(
    network_path,
    device,
    gen_nonsquare,
    width,
    height
):

    with suppress_output():

        with dnnlib.util.open_url(
            network_path
        ) as f:


            # ------------------------------------------------
            # GENERACIÓN RECTANGULAR INTERNA
            # ------------------------------------------------

            if gen_nonsquare:

                data = legacy.load_network_pkl(
                    f,

                    # Activa stylegan2_multi.py
                    custom=True,

                    # Internamente: [alto, ancho]
                    size=[
                        height,
                        width
                    ],

                    # Método original del fork
                    scale_type="symm"
                )


            # ------------------------------------------------
            # GENERACIÓN NORMAL CUADRADA
            # ------------------------------------------------

            else:

                data = legacy.load_network_pkl(
                    f
                )


    G = data[
        "G_ema"
    ].to(device)

    G.eval()
    G.requires_grad_(False)

    return G


# ============================================================
# GENERACIÓN
# ============================================================

def generate_images(b):

    clear_output(
        wait=True
    )

    display(ui)


    with output_area:

        clear_output(
            wait=True
        )

        print(
            "Preparando generación...\n"
        )


        # ----------------------------------------------------
        # Parámetros
        # ----------------------------------------------------

        outdir = (
            outdir_widget
            .value
            .strip()
        )


        network_path = (
            network_widget
            .value
            .strip()
        )


        truncation = (
            trunc_widget.value
        )


        noise_mode = (
            noise_widget.value
        )


        gen_random = (
            gen_random_widget.value
        )


        n_imgs = (
            n_imgs_widget.value
        )


        seeds_input = (
            seeds_widget
            .value
            .strip()
        )


        gen_nonsquare = (
            gen_nonsquare_widget.value
        )


        width = int(
            width_widget.value
        )


        height = int(
            height_widget.value
        )


        save_images = (
            save_widget.value
        )


        show_preview = (
            preview_widget.value
        )


        preview_size = (
            preview_size_widget.value
        )


        # ----------------------------------------------------
        # Comprobar modelo
        # ----------------------------------------------------

        if network_path == "":

            print(
                "Debes indicar un archivo .pkl "
                "o usar el botón de buscar snapshot."
            )

            return


        # ----------------------------------------------------
        # Comprobar dimensiones
        # ----------------------------------------------------

        if gen_nonsquare:

            if (
                width <= 0
                or height <= 0
            ):

                print(
                    "El ancho y el alto deben "
                    "ser mayores que 0."
                )

                return


        # ----------------------------------------------------
        # Preparar salida
        # ----------------------------------------------------

        os.makedirs(
            outdir,
            exist_ok=True
        )


        device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )


        print(
            f"Cargando modelo en: {device}"
        )


        # ----------------------------------------------------
        # Cargar modelo
        # ----------------------------------------------------

        try:

            G = load_generator(
                network_path=network_path,
                device=device,
                gen_nonsquare=gen_nonsquare,
                width=width,
                height=height
            )

        except Exception as e:

            print()
            print(
                "Error cargando el modelo:"
            )

            print(e)

            return


        print(
            "Modelo cargado correctamente."
        )


        if gen_nonsquare:

            print(
                f"Síntesis interna: "
                f"{width} × {height} px"
            )

            print(
                "Scale type: symm"
            )

        else:

            print(
                f"Resolución: "
                f"{G.img_resolution} × "
                f"{G.img_resolution}"
            )


        print()


        # ----------------------------------------------------
        # Preparar seeds
        # ----------------------------------------------------

        if gen_random:

            seeds_list = [

                int(seed)

                for seed in np.random.randint(
                    0,
                    4294967295,
                    size=n_imgs,
                    dtype=np.uint32
                )
            ]


        else:

            try:

                seeds_list = parse_seeds(
                    seeds_input
                )

            except Exception as e:

                print(
                    f"Error en los seeds: {e}"
                )

                return


        if not seeds_list:

            print(
                "No hay seeds para generar."
            )

            return


        print(
            f"Seeds a generar: {seeds_list}"
        )

        print(
            f"Truncation ψ: {truncation}"
        )

        print(
            f"Noise mode: {noise_mode}"
        )

        print()


        # ----------------------------------------------------
        # Generar
        # ----------------------------------------------------

        generated = []


        with torch.inference_mode():


            for seed in seeds_list:


                rnd = np.random.RandomState(
                    seed
                )


                z = rnd.randn(
                    1,
                    G.z_dim
                ).astype(
                    np.float32
                )


                z_tensor = torch.from_numpy(
                    z
                ).to(
                    device=device,
                    dtype=torch.float32
                )


                label = make_label(
                    G,
                    device
                )


                img_tensor = G(
                    z_tensor,
                    label,
                    truncation_psi=truncation,
                    noise_mode=noise_mode
                )


                # --------------------------------------------
                # IMPORTANTE:
                #
                # Si gen_nonsquare=True, la salida ya viene
                # directamente del generador en width × height.
                #
                # No hay resize, crop ni np.pad posterior.
                # --------------------------------------------

                pil_img = tensor_to_pil(
                    img_tensor
                )


                # --------------------------------------------
                # Nombre del archivo
                # --------------------------------------------

                if gen_nonsquare:

                    filename = os.path.join(
                        outdir,
                        (
                            f"img_seed_{seed}_"
                            f"{width}x{height}.png"
                        )
                    )


                else:

                    filename = os.path.join(
                        outdir,
                        f"img_seed_{seed}.png"
                    )


                # --------------------------------------------
                # Guardar
                # --------------------------------------------

                if save_images:

                    pil_img.save(
                        filename
                    )

                    print(
                        f"Guardada: {filename}"
                    )

                else:

                    print(
                        f"Generada: seed {seed}"
                    )


                generated.append(
                    (
                        seed,
                        pil_img
                    )
                )


        print()
        print(
            "Generación completada."
        )


        # ----------------------------------------------------
        # Vista previa
        # ----------------------------------------------------

        if (
            show_preview
            and generated
        ):

            print()
            print(
                "Vista previa:"
            )


            for seed, img in generated:

                print()
                print(
                    f"Seed: {seed}"
                )

                print(
                    f"Resolución: "
                    f"{img.size[0]} × {img.size[1]}"
                )


                preview_img = (
                    make_preview_image(
                        img,
                        preview_size=preview_size
                    )
                )


                display(
                    preview_img
                )


generate_button.on_click(
    generate_images
)


# ============================================================
# ORGANIZACIÓN DE LA INTERFAZ
# ============================================================

section_model = widgets.VBox([
    project_dir_widget,

    widgets.HBox([
        find_latest_button
    ]),

    network_widget
])


section_gen = widgets.VBox([

    trunc_widget,

    noise_widget,

    seeds_widget,

    widgets.HBox([
        gen_random_widget,
        n_imgs_widget
    ]),

    widgets.HBox([
        save_widget,
        preview_widget
    ]),

    preview_size_widget
])


section_size = widgets.VBox([

    gen_nonsquare_widget,

    widgets.HBox([
        width_widget,
        height_widget
    ])
])


ui = widgets.VBox([

    section_model,

    outdir_widget,

    section_gen,

    section_size,

    generate_button,

    output_area
])


display(ui)

In [30]:
#@title 6. Explorador de Espacio Latente StyleGAN2

import os
import sys
import io
import warnings
import contextlib
import numpy as np
import torch

from pathlib import Path
from PIL import Image
from IPython.display import display
import ipywidgets as widgets

import dnnlib
import legacy


# ============================================================
# CONFIGURACIÓN
# ============================================================

warnings.filterwarnings("ignore")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

G = None
latent_dim = 512
num_directions = 5

last_image = None


@contextlib.contextmanager
def suppress_output():

    with open(os.devnull, "w") as devnull:

        old_stdout = sys.stdout
        old_stderr = sys.stderr

        sys.stdout = devnull
        sys.stderr = devnull

        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


# ============================================================
# DIRECCIONES LATENTES
# ============================================================

direction_rng = np.random.RandomState(2026)

slider_matrix = direction_rng.randn(
    num_directions,
    latent_dim
).astype(np.float32)

slider_matrix /= np.linalg.norm(
    slider_matrix,
    axis=1,
    keepdims=True
)


# ============================================================
# MODELO Y SALIDA
# ============================================================

network_widget = widgets.Text(
    value="",
    description="Modelo:",
    layout=widgets.Layout(width="460px")
)

output_dir_widget = widgets.Text(
    value="/content/drive/MyDrive/colab-sg2-ada-pytorch/generated/explatent/",
    description="Salida:",
    layout=widgets.Layout(width="460px")
)

load_model_button = widgets.Button(
    description="Cargar modelo",
    button_style="info",
    layout=widgets.Layout(width="140px")
)

save_button = widgets.Button(
    description="Guardar imagen",
    button_style="success",
    layout=widgets.Layout(width="140px")
)

status_widget = widgets.HTML(
    value="<small>Modelo no cargado</small>"
)


# ============================================================
# SEEDS
# ============================================================

seed_a_widget = widgets.IntText(
    value=500,
    description="Seed A:",
    layout=widgets.Layout(width="150px")
)

seed_b_widget = widgets.IntText(
    value=501,
    description="Seed B:",
    layout=widgets.Layout(width="150px")
)

seed_c_widget = widgets.IntText(
    value=502,
    description="Seed C:",
    layout=widgets.Layout(width="150px")
)

interpolation_widget = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=2.0,
    step=0.01,
    description="Interpolación:",
    continuous_update=True,
    readout_format=".2f",
    layout=widgets.Layout(width="420px")
)

trunc_widget = widgets.FloatSlider(
    value=0.8,
    min=0.1,
    max=1.5,
    step=0.01,
    description="Trunc ψ:",
    continuous_update=True,
    readout_format=".2f",
    layout=widgets.Layout(width="420px")
)


# ============================================================
# DIRECCIONES DE EXPLORACIÓN
# ============================================================

direction_sliders = {}

for i in range(num_directions):

    direction_sliders[str(i)] = widgets.FloatSlider(
        value=0.0,
        min=-5.0,
        max=5.0,
        step=0.05,
        description=f"Dim. {i + 1}:",
        continuous_update=True,
        readout_format=".2f",
        layout=widgets.Layout(width="420px")
    )

reset_button = widgets.Button(
    description="Restablecer",
    layout=widgets.Layout(width="120px")
)


# ============================================================
# VISUALIZACIÓN
# ============================================================

image_widget = widgets.Image(
    format="jpeg",
    width=560,
    height=560
)

position_widget = widgets.HTML(
    value=""
)

message_output = widgets.Output(
    layout=widgets.Layout(
        width="420px",
        max_height="100px"
    )
)


# ============================================================
# CARGAR MODELO
# ============================================================

def load_model(b=None):

    global G
    global latent_dim
    global slider_matrix

    network_path = network_widget.value.strip()

    with message_output:

        message_output.clear_output()

        if not network_path:
            print("Indica la ruta de un modelo .pkl.")
            return

        print("Cargando modelo...")

    try:

        with suppress_output():

            with dnnlib.util.open_url(
                network_path
            ) as f:

                G = legacy.load_network_pkl(
                    f
                )["G_ema"].to(device)

        G.eval()
        G.requires_grad_(False)

        latent_dim = G.z_dim

        rng = np.random.RandomState(2026)

        slider_matrix = rng.randn(
            num_directions,
            latent_dim
        ).astype(np.float32)

        slider_matrix /= np.linalg.norm(
            slider_matrix,
            axis=1,
            keepdims=True
        )

        status_widget.value = (
            f"<small>"
            f"Modelo cargado · "
            f"{G.img_resolution} × {G.img_resolution} · "
            f"{device}"
            f"</small>"
        )

        with message_output:
            message_output.clear_output()

        render_image()

    except Exception as e:

        status_widget.value = (
            "<small>Error al cargar modelo</small>"
        )

        with message_output:
            message_output.clear_output()
            print(e)


load_model_button.on_click(
    load_model
)


# ============================================================
# SEED -> Z
# ============================================================

def seed_to_z(seed):

    rng = np.random.RandomState(
        int(seed)
    )

    return rng.randn(
        latent_dim
    ).astype(np.float32)


# ============================================================
# INTERPOLACIÓN ESFÉRICA
# ============================================================

def slerp(z1, z2, t):

    z1 = z1.astype(np.float32)
    z2 = z2.astype(np.float32)

    norm1 = np.linalg.norm(z1)
    norm2 = np.linalg.norm(z2)

    if norm1 == 0 or norm2 == 0:

        return (
            (1.0 - t) * z1
            + t * z2
        ).astype(np.float32)

    u1 = z1 / norm1
    u2 = z2 / norm2

    dot = np.clip(
        np.dot(u1, u2),
        -1.0,
        1.0
    )

    if abs(dot) > 0.9995:

        return (
            (1.0 - t) * z1
            + t * z2
        ).astype(np.float32)

    theta = np.arccos(dot)
    sin_theta = np.sin(theta)

    direction = (
        np.sin((1.0 - t) * theta) / sin_theta * u1
        +
        np.sin(t * theta) / sin_theta * u2
    )

    radius = (
        (1.0 - t) * norm1
        + t * norm2
    )

    return (direction * radius).astype(np.float32)


# ============================================================
# INTERPOLACIÓN ENTRE 3 SEEDS
# ============================================================

def interpolate_three_seeds():

    z_a = seed_to_z(seed_a_widget.value)
    z_b = seed_to_z(seed_b_widget.value)
    z_c = seed_to_z(seed_c_widget.value)

    t = interpolation_widget.value

    if t <= 1.0:
        # tramo A -> B
        return slerp(z_a, z_b, t)
    else:
        # tramo B -> C
        return slerp(z_b, z_c, t - 1.0)


# ============================================================
# POSICIÓN ACTUAL
# ============================================================

def get_current_z():

    z = interpolate_three_seeds()

    for i in range(num_directions):

        z += (
            direction_sliders[str(i)].value
            * slider_matrix[i]
        )

    return z.astype(np.float32)


# ============================================================
# GENERACIÓN
# ============================================================

def make_label():

    if getattr(G, "c_dim", 0) > 0:

        return torch.zeros(
            [1, G.c_dim],
            device=device
        )

    return None


def generate_image(z):

    z_tensor = torch.from_numpy(
        z
    ).to(
        device=device,
        dtype=torch.float32
    ).unsqueeze(0)

    with torch.inference_mode():

        img_tensor = G(
            z_tensor,
            make_label(),
            truncation_psi=trunc_widget.value,
            noise_mode="const"
        )

    img = (
        (img_tensor.clamp(-1, 1) + 1)
        * 127.5
    )

    img = (
        img
        .permute(0, 2, 3, 1)
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

    return Image.fromarray(
        img[0],
        "RGB"
    )


# ============================================================
# VISUALIZACIÓN
# ============================================================

def image_to_widget_bytes(img):

    view = img.copy()

    view.thumbnail(
        (560, 560),
        Image.LANCZOS
    )

    buffer = io.BytesIO()

    view.save(
        buffer,
        format="JPEG",
        quality=90
    )

    return buffer.getvalue()


# ============================================================
# ACTUALIZAR IMAGEN
# ============================================================

def render_image():

    global last_image

    if G is None:
        return

    z = get_current_z()

    last_image = generate_image(z)

    image_widget.value = image_to_widget_bytes(
        last_image
    )

    t = interpolation_widget.value

    if t <= 1.0:
        tramo = f"Seed {seed_a_widget.value} → {seed_b_widget.value}"
        local_t = t
    else:
        tramo = f"Seed {seed_b_widget.value} → {seed_c_widget.value}"
        local_t = t - 1.0

    position_widget.value = (
        f"<small>"
        f"{tramo}"
        f" &nbsp;|&nbsp; "
        f"posición global: {t:.2f}"
        f" &nbsp;|&nbsp; "
        f"posición local: {local_t:.2f}"
        f"</small>"
    )


# ============================================================
# ACTUALIZACIÓN AUTOMÁTICA
# ============================================================

def schedule_render(change=None):

    if G is None:
        return

    render_image()


# ============================================================
# RESTABLECER
# ============================================================

def reset_controls(b=None):

    interpolation_widget.value = 0.0

    for slider in direction_sliders.values():
        slider.value = 0.0

    render_image()


reset_button.on_click(
    reset_controls
)


# ============================================================
# GUARDAR IMAGEN
# ============================================================

def save_current_image(b=None):

    if G is None or last_image is None:

        with message_output:

            message_output.clear_output()

            print("Carga primero un modelo.")

        return

    output_dir = Path(
        output_dir_widget.value.strip()
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    filename = (
        output_dir
        /
        (
            f"seed_{seed_a_widget.value}_"
            f"seed_{seed_b_widget.value}_"
            f"seed_{seed_c_widget.value}_"
            f"t_{interpolation_widget.value:.2f}.png"
        )
    )

    last_image.save(filename)

    with message_output:

        message_output.clear_output()

        print("Imagen guardada:")
        print(filename)


save_button.on_click(
    save_current_image
)


# ============================================================
# EVENTOS
# ============================================================

seed_a_widget.observe(
    schedule_render,
    names="value"
)

seed_b_widget.observe(
    schedule_render,
    names="value"
)

seed_c_widget.observe(
    schedule_render,
    names="value"
)

interpolation_widget.observe(
    schedule_render,
    names="value"
)

trunc_widget.observe(
    schedule_render,
    names="value"
)

for slider in direction_sliders.values():

    slider.observe(
        schedule_render,
        names="value"
    )


# ============================================================
# INTERFAZ
# ============================================================

controls = widgets.VBox(
    [
        widgets.HTML("<b>Modelo</b>"),

        network_widget,
        output_dir_widget,

        widgets.HBox([
            load_model_button,
            save_button
        ]),

        status_widget,

        widgets.HTML("<hr>"),

        widgets.HTML("<b>Interpolación</b>"),

        widgets.HBox([
            seed_a_widget,
            seed_b_widget,
            seed_c_widget
        ]),

        interpolation_widget,
        trunc_widget,

        widgets.HTML("<hr>"),

        widgets.HTML("<b>Exploración latente</b>"),

        *direction_sliders.values(),

        reset_button,

        message_output
    ],

    layout=widgets.Layout(
        width="460px",
        min_width="460px"
    )
)

viewer = widgets.VBox(
    [
        image_widget,
        position_widget
    ],

    layout=widgets.Layout(
        width="620px",
        align_items="center",
        margin="0 0 0 50px"
    )
)

ui = widgets.HBox(
    [
        controls,
        viewer
    ],

    layout=widgets.Layout(
        width="1100px",
        align_items="flex-start",
        justify_content="flex-start",
        gap="40px"
    )
)

display(ui)

In [ ]:
#@title 7. Interpolación de Seeds StyleGAN2 → Video

import os
import sys
import re
import contextlib
import warnings

import numpy as np
import torch
import imageio.v2 as imageio

from tqdm.auto import tqdm
from IPython.display import display, Video

import dnnlib
import legacy


# ============================================================
# PARÁMETROS
# ============================================================

# Modelo StyleGAN2
network_pkl = "/content/drive/MyDrive/00070-amonites-11gb-gpu-noaug-resumecustom/network-snapshot-000020.pkl" #@param {type:"string"}

# Ruta completa del video
video_path = "/content/drive/MyDrive/colab-sg2-ada-pytorch/generated/interpolacion5.mp4" #@param {type:"string"}


# ------------------------------------------------------------
# SEEDS
# ------------------------------------------------------------

# Ejemplos:
# 498,499,500
# 500-510
# 1200
seeds_input = "498,499,500" #@param {type:"string"}

# Si está activado, seeds_input será ignorado.
usar_seeds_aleatorios = False #@param {type:"boolean"}

cantidad_seeds_aleatorios = 3 #@param {type:"integer"}


# ------------------------------------------------------------
# VIDEO
# ------------------------------------------------------------

# Duración TOTAL del video
duracion = 20 #@param {type:"number"}

fps = 24 #@param {type:"integer"}

truncation = 0.8 #@param {type:"number"}


# ------------------------------------------------------------
# FORMATO
# ------------------------------------------------------------

# Si está activado, StyleGAN sintetiza directamente
# en formato rectangular usando scale_type="symm".
formato_no_cuadrado = True #@param {type:"boolean"}

ancho = 1920 #@param {type:"integer"}

alto = 1080 #@param {type:"integer"}


# ============================================================
# CONFIGURACIÓN
# ============================================================

warnings.filterwarnings("ignore")


@contextlib.contextmanager
def suppress_output():

    with open(os.devnull, "w") as devnull:

        old_stdout = sys.stdout
        old_stderr = sys.stderr

        sys.stdout = devnull
        sys.stderr = devnull

        try:
            yield

        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


# ============================================================
# LEER SEEDS
# ============================================================

def parse_seeds(text):

    text = (
        text
        .strip()
        .replace(" ", "")
    )

    if text == "":

        raise ValueError(
            "Debes ingresar al menos un seed."
        )


    # --------------------------------------------------------
    # Rango: 500-510
    # --------------------------------------------------------

    if re.fullmatch(
        r"\d+-\d+",
        text
    ):

        start, end = map(
            int,
            text.split("-")
        )

        if end >= start:

            return list(
                range(
                    start,
                    end + 1
                )
            )

        else:

            return list(
                range(
                    start,
                    end - 1,
                    -1
                )
            )


    # --------------------------------------------------------
    # Lista o seed individual
    # --------------------------------------------------------

    try:

        return [
            int(seed)
            for seed
            in text.split(",")
            if seed != ""
        ]

    except ValueError:

        raise ValueError(
            "Formato de seeds inválido."
        )


# ============================================================
# OBTENER SEEDS
# ============================================================

def get_seeds():

    if usar_seeds_aleatorios:

        if cantidad_seeds_aleatorios < 2:

            raise ValueError(
                "Para una interpolación se necesitan "
                "al menos 2 seeds aleatorios."
            )

        rng = np.random.default_rng()

        seeds = rng.integers(
            0,
            2**32 - 1,
            size=cantidad_seeds_aleatorios,
            dtype=np.uint32
        )

        return [
            int(seed)
            for seed in seeds
        ]


    return parse_seeds(
        seeds_input
    )


# ============================================================
# SEED -> Z
# ============================================================

def seed_to_z(
    seed,
    z_dim
):

    rng = np.random.RandomState(
        int(seed)
    )

    return rng.randn(
        z_dim
    ).astype(np.float32)


# ============================================================
# INTERPOLACIÓN ESFÉRICA
# ============================================================

def slerp(
    z1,
    z2,
    t
):

    z1 = z1.astype(np.float32)
    z2 = z2.astype(np.float32)

    norm1 = np.linalg.norm(z1)
    norm2 = np.linalg.norm(z2)

    if (
        norm1 == 0
        or norm2 == 0
    ):

        return (
            (1.0 - t) * z1
            + t * z2
        ).astype(np.float32)


    u1 = z1 / norm1
    u2 = z2 / norm2


    dot = np.clip(
        np.dot(
            u1,
            u2
        ),
        -1.0,
        1.0
    )


    if abs(dot) > 0.9995:

        return (
            (1.0 - t) * z1
            + t * z2
        ).astype(np.float32)


    theta = np.arccos(
        dot
    )

    sin_theta = np.sin(
        theta
    )


    direction = (

        np.sin(
            (1.0 - t)
            * theta
        )
        / sin_theta
        * u1

        +

        np.sin(
            t * theta
        )
        / sin_theta
        * u2
    )


    radius = (
        (1.0 - t) * norm1
        + t * norm2
    )


    return (
        direction
        * radius
    ).astype(np.float32)


# ============================================================
# DIMENSIONES PARES PARA H.264
# ============================================================

def even_dimension(
    value
):

    value = int(
        value
    )

    if value <= 0:

        raise ValueError(
            "Las dimensiones deben ser mayores que 0."
        )

    # yuv420p requiere dimensiones pares.
    if value % 2 != 0:
        value -= 1

    return max(
        value,
        2
    )


# ============================================================
# DISPOSITIVO
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# RESOLUCIÓN SOLICITADA
# ============================================================

if formato_no_cuadrado:

    final_width = even_dimension(
        ancho
    )

    final_height = even_dimension(
        alto
    )

else:

    final_width = None
    final_height = None


# ============================================================
# CARGAR GENERADOR
# ============================================================

print("Cargando modelo...")


with suppress_output():

    with dnnlib.util.open_url(
        network_pkl
    ) as f:


        # ----------------------------------------------------
        # GENERADOR RECTANGULAR INTERNO
        # ----------------------------------------------------

        if formato_no_cuadrado:

            data = legacy.load_network_pkl(
                f,

                custom=True,

                # Internamente usa [alto, ancho]
                size=[
                    final_height,
                    final_width
                ],

                scale_type="symm"
            )


        # ----------------------------------------------------
        # GENERADOR NORMAL
        # ----------------------------------------------------

        else:

            data = legacy.load_network_pkl(
                f
            )


G = data[
    "G_ema"
].to(device)

G.eval()
G.requires_grad_(False)


print("Modelo cargado.")

print(
    f"Dispositivo: {device}"
)


if formato_no_cuadrado:

    print(
        f"Síntesis interna: "
        f"{final_width} × {final_height}"
    )

    print(
        "Scale type: symm"
    )

else:

    final_width = G.img_resolution
    final_height = G.img_resolution

    print(
        f"Resolución: "
        f"{final_width} × {final_height}"
    )


# ============================================================
# PREPARAR SEEDS
# ============================================================

seeds = get_seeds()


if len(seeds) < 2:

    raise ValueError(
        "Se necesitan al menos 2 seeds "
        "para crear una interpolación."
    )


print()
print(
    "Seeds utilizados:"
)

print(
    seeds
)


# ============================================================
# CREAR VECTORES LATENTES
# ============================================================

latents = [

    seed_to_z(
        seed,
        G.z_dim
    )

    for seed in seeds

]


# ============================================================
# CONFIGURACIÓN DEL VIDEO
# ============================================================

if duracion <= 0:

    raise ValueError(
        "La duración debe ser mayor que 0."
    )


if fps <= 0:

    raise ValueError(
        "FPS debe ser mayor que 0."
    )


frames_total = int(
    round(
        duracion
        * fps
    )
)


if frames_total < 2:

    raise ValueError(
        "La combinación de duración y FPS "
        "debe producir al menos 2 frames."
    )


num_segments = (
    len(latents)
    - 1
)


seconds_per_segment = (
    duracion
    / num_segments
)


print()

print(
    f"Duración total: "
    f"{duracion} s"
)

print(
    f"FPS: "
    f"{fps}"
)

print(
    f"Frames totales: "
    f"{frames_total}"
)

print(
    f"Seeds: "
    f"{len(seeds)}"
)

print(
    f"Transiciones: "
    f"{num_segments}"
)

print(
    "Duración aproximada por transición: "
    f"{seconds_per_segment:.2f} s"
)

print(
    f"Resolución del video: "
    f"{final_width} × {final_height}"
)

print()


# ============================================================
# LABEL
# ============================================================

if getattr(
    G,
    "c_dim",
    0
) > 0:

    label = torch.zeros(
        [
            1,
            G.c_dim
        ],
        device=device
    )

else:

    label = None


# ============================================================
# GENERAR FRAME
# ============================================================

def generate_frame(
    z
):

    z_tensor = torch.from_numpy(
        z
    ).to(
        device=device,
        dtype=torch.float32
    ).unsqueeze(0)


    with torch.inference_mode():

        img_tensor = G(
            z_tensor,
            label,
            truncation_psi=truncation,
            noise_mode="const"
        )


    img = (
        (
            img_tensor.clamp(
                -1,
                1
            )
            + 1
        )
        * 127.5
    )


    img = (
        img
        .permute(
            0,
            2,
            3,
            1
        )
        .cpu()
        .numpy()
        .astype(np.uint8)
    )


    # IMPORTANTE:
    #
    # No hay resize, crop ni np.pad.
    #
    # Cuando formato_no_cuadrado=True,
    # esta imagen ya fue sintetizada por
    # StyleGAN directamente con las
    # dimensiones rectangulares.

    return img[0]


# ============================================================
# PREPARAR RUTA
# ============================================================

output_folder = os.path.dirname(
    video_path
)


if output_folder:

    os.makedirs(
        output_folder,
        exist_ok=True
    )


# ============================================================
# GENERAR VIDEO
# ============================================================

print(
    "Generando video..."
)


writer = imageio.get_writer(
    video_path,
    fps=fps,
    codec="libx264",
    quality=8,
    output_params=[
        "-pix_fmt",
        "yuv420p"
    ]
)


try:

    for frame_index in tqdm(
        range(
            frames_total
        ),
        desc="Frames"
    ):


        # ----------------------------------------------------
        # Posición global a través de TODOS los seeds
        # ----------------------------------------------------

        global_position = (

            frame_index
            / (
                frames_total
                - 1
            )

            * num_segments
        )


        # ----------------------------------------------------
        # Último frame
        # ----------------------------------------------------

        if (
            frame_index
            == frames_total - 1
        ):

            segment = (
                num_segments
                - 1
            )

            local_t = 1.0


        else:

            segment = int(
                np.floor(
                    global_position
                )
            )

            local_t = (
                global_position
                - segment
            )


        # ----------------------------------------------------
        # Seeds correspondientes al segmento
        # ----------------------------------------------------

        z_start = latents[
            segment
        ]

        z_end = latents[
            segment + 1
        ]


        # ----------------------------------------------------
        # SLERP
        # ----------------------------------------------------

        z = slerp(
            z_start,
            z_end,
            local_t
        )


        # ----------------------------------------------------
        # Generar frame
        # ----------------------------------------------------

        frame = generate_frame(
            z
        )


        # ----------------------------------------------------
        # Seguridad
        # ----------------------------------------------------

        if (
            frame.shape[1]
            != final_width
            or
            frame.shape[0]
            != final_height
        ):

            raise RuntimeError(
                "El generador produjo una resolución "
                "inesperada: "
                f"{frame.shape[1]} × {frame.shape[0]} "
                f"en lugar de "
                f"{final_width} × {final_height}."
            )


        writer.append_data(
            frame
        )


finally:

    writer.close()


# ============================================================
# RESULTADO
# ============================================================

print()

print(
    "Video generado:"
)

print(
    video_path
)

print()

print(
    "Seeds utilizados:"
)

print(
    seeds
)


# ============================================================
# PREVISUALIZAR EN COLAB
# ============================================================

print()
print(
    "Vista previa:"
)


display(
    Video(
        video_path,
        embed=True,
        html_attributes=(
            "controls "
            "playsinline "
            "style='max-width:900px; width:100%;'"
        )
    )
)